# 05 — asyncio Event Loops, Coroutines, and Awaitables

asyncio coordinates I/O-oriented concurrency through cooperative suspension. Distinguish coroutine objects, awaited results, tasks, and event-loop lifecycle.

*Advanced · 90–150 min*

## You will learn
- Write and await asynchronous functions
- Explain event-loop scheduling and task creation
- Compare sequential awaits with concurrent gather
- Use asyncio.run appropriately and avoid blocking the event loop

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. Coroutines are not results

Calling an async def function creates a coroutine object; its body executes when awaited or scheduled as a task. asyncio.run is a top-level entry point in ordinary Python scripts, not something to nest inside an already running event loop.

In [ ]:
from lesson import run  # like asyncio.run(), but also works inside Jupyter and Colab
import asyncio

async def greeting(name):
    await asyncio.sleep(0)
    return f"Hello, {name}"

async def demo():
    return await greeting("Ada")

print(run(demo()))

**Think:** How would a Jupyter environment with a running loop differ from an ordinary script?

### 2. Sequential vs concurrent I/O

await suspends one coroutine to let other tasks proceed, but sequential awaits still perform operations in order. asyncio.gather schedules several awaitables together and returns their results in input order.

In [ ]:
from lesson import run  # like asyncio.run(), but also works inside Jupyter and Colab
import asyncio
async def worker(label):
    await asyncio.sleep(0.001)
    return label.upper()

async def demo():
    serial = [await worker("a"), await worker("b")]
    parallel = await asyncio.gather(worker("a"), worker("b"))
    return serial, parallel

print(run(demo()))

**Think:** Why is gather not expected to speed up CPU-bound Python computations?

### 3. Tasks and explicit synchronization

create_task schedules a coroutine on the current loop; tasks may start running before they are awaited. Always retain references to tasks and eventually await them, otherwise failures can be lost or work abandoned.

In [ ]:
from lesson import run  # like asyncio.run(), but also works inside Jupyter and Colab
import asyncio
async def child(value):
    await asyncio.sleep(0)
    return value * 2

async def parent():
    task = asyncio.create_task(child(21), name="double")
    print("Scheduled:", task.get_name())
    return await task

print(run(parent()))

**Think:** What lifecycle problems arise from fire-and-forget tasks?

### 4. Never block the loop with synchronous sleep

time.sleep and blocking network calls halt the thread running the event loop. Use asyncio.sleep for cooperative waiting and asyncio.to_thread for appropriate blocking operations. Do not claim asyncio makes a function intrinsically faster.

In [ ]:
from lesson import run  # like asyncio.run(), but also works inside Jupyter and Colab
import asyncio
import time

def blocking_lookup(value):
    return value * value

async def safe_demo():
    start = time.perf_counter()
    result = await asyncio.to_thread(blocking_lookup, 12)
    return result, time.perf_counter() - start

value, elapsed = run(safe_demo())
print(value, isinstance(elapsed, float))

**Think:** Why is to_thread a boundary for blocking I/O rather than a blanket CPU speedup?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/02_concurrency_resilience_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/05', {1: '6d98b90fa702272f', 2: 'bdf79c307c33b89c', 3: '0e9f10cceb3e624d', 4: '55503ac3b3caea2b'})

**Exercise 1.** Three async workers; collect results in the order the tasks were created.

In [ ]:
import asyncio

async def worker(n):
    await asyncio.sleep(0.01 * (3 - n))     # later tasks finish first
    return n * 10

async def main():
    tasks = [asyncio.create_task(worker(n)) for n in range(3)]
    return [___ for t in tasks]

check(1, run(main()))

**Exercise 2.** `gather` overlaps waits; awaiting one by one doesn't. Results are the same.

In [ ]:
import asyncio, time

async def nap(x):
    await asyncio.sleep(0.05)
    return x

async def sequential():
    return [await nap(i) for i in range(4)]

async def concurrent():
    return await asyncio.___(*(nap(i) for i in range(4)))

start = time.perf_counter(); seq = run(sequential()); t_seq = time.perf_counter() - start
start = time.perf_counter(); con = run(concurrent()); t_con = time.perf_counter() - start
check(2, (seq == list(con), t_con < t_seq))

**Exercise 3.** Move blocking work off the event loop with `asyncio.to_thread`.

In [ ]:
import asyncio, threading

def blocking_read():
    return threading.current_thread().name != "MainThread"

async def main():
    return await asyncio.___(blocking_read)

check(3, run(main()))

**Exercise 4.** A coroutine object does nothing until awaited; a Task starts running on its own.

In [ ]:
import asyncio

log = []
async def job(name):
    log.append(name)

async def main():
    coro = job("coroutine")              # created, not started
    task = asyncio.___(job("task"))
    await asyncio.sleep(0)               # let the loop run
    started = list(log)
    await coro
    await task
    return started

check(4, run(main()))

## Common mistakes

- Do not call asyncio.run from inside an active event loop.
- Do not create tasks that you never await or otherwise supervise.
- Blocking synchronous operations can stall all cooperative tasks.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Calling an async def function returns…

   - **a)** A thread
   - **b)** Its result
   - **c)** A coroutine object that must be awaited

**2.** What does time.sleep(1) do inside a coroutine?

   - **a)** Yields to other tasks
   - **b)** Nothing
   - **c)** Blocks the whole event loop

**3.** asyncio.gather(a(), b()) runs a and b…

   - **a)** Concurrently
   - **b)** In separate processes
   - **c)** One after the other

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/05', {1: 'e64c9c28fd99', 2: '9bded35b3584', 3: '0ca6aa20463f'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [06 — TaskGroup, Cancellation, and Structured Concurrency](06_taskgroups_cancellation_timeouts.ipynb)